# Festa em Cena — análise de vendas Halloween

**Sprint 1:** configuração, validação do arquivo e ingestão com `csv.DictReader`.

Cada **linha** do CSV é um pedido de um único produto (pode haver várias unidades em `quantidade`). Os horários em `data_venda` são locais (São Paulo), sem fuso explícito.

## Significado de colunas (amostra)

| Coluna | Significado |
|--------|-------------|
| `pedido_id` | Identificador do pedido |
| `data_venda` | Data/hora do registro da venda |
| `canal_venda` | Mercado, Loja física ou Web |
| `status_pedido` | Situação do pedido (ex.: Concluído, Cancelado) |

In [ ]:
from pathlib import Path

from functions import (
    DEFAULT_CSV_NAME,
    EXPECTED_COLUMNS,
    default_csv_path,
    read_sales_csv,
    validate_csv_file,
)

csv_path = default_csv_path()
print(f"Arquivo esperado: {csv_path}")
print(f"Existe no disco: {csv_path.is_file()}")

## Validação estrutural do CSV

Antes de analisar os dados, verificamos existência, UTF-8, cabeçalho e colunas obrigatórias.

In [ ]:
validation = validate_csv_file(csv_path)
validation.summary()

In [ ]:
if not validation.ok:
    raise SystemExit("Corrija o arquivo CSV antes de continuar.")

print(f"Colunas esperadas ({len(EXPECTED_COLUMNS)}): {', '.join(EXPECTED_COLUMNS)}")
print(f"Total de linhas de dados: {validation.row_count}")

## Leitura com `csv.DictReader`

Todos os campos são mantidos como **texto**; conversões numéricas e de data ficam para sprints posteriores.

In [ ]:
registros = read_sales_csv(csv_path)
len(registros), list(registros[0].keys())

In [ ]:
print("Três registros de exemplo:\n")
for i, row in enumerate(registros[:3], start=1):
    print(f"--- Registro {i} (pedido_id={row['pedido_id']}) ---")
    for chave in EXPECTED_COLUMNS:
        print(f"  {chave}: {row[chave]}")
    print()

## Padronização de strings

Limpeza de `categoria` com `.strip()`, `.lower()` e `re.sub()` (pontuação removida; letras e espaços preservados). Canais reconhecidos passam a **Mercado**, **Loja física** e **Web**. O arquivo CSV no disco não é alterado.

In [ ]:
from functions import transform_string_fields

registros_limpos, stats_strings = transform_string_fields(registros)
stats_strings.summary()

In [ ]:
def exemplos_antes_depois(
    original: list[dict[str, str]],
    transformado: list[dict[str, str]],
    campo: str,
    limite: int = 3,
) -> list[tuple[str, str]]:
    pares: list[tuple[str, str]] = []
    for antes, depois in zip(original, transformado):
        if antes[campo] != depois[campo]:
            pares.append((antes[campo], depois[campo]))
        if len(pares) >= limite:
            break
    return pares

print("Categoria (até 3 exemplos):")
for antes, depois in exemplos_antes_depois(registros, registros_limpos, "categoria"):
    print(f"  {antes!r} -> {depois!r}")

print("\nCanal de venda (até 3 exemplos):")
for antes, depois in exemplos_antes_depois(registros, registros_limpos, "canal_venda"):
    print(f"  {antes!r} -> {depois!r}")

## Validação numérica (quantidade e preço)

Verificamos `quantidade` e `preco_unitario` sem corrigir valores. Registros com problema são separados em outra lista para análise; o fluxo continua para todos os pedidos.

In [ ]:
from functions import validate_numeric_fields

validos, com_problema, stats_numericos = validate_numeric_fields(registros_limpos)
stats_numericos.summary()

In [ ]:
print(f"Registros válidos: {len(validos)}")
print(f"Registros com problema: {len(com_problema)}")
print(f"Pedidos distintos afetados: {stats_numericos.pedidos_com_problema}\n")

print("Até 3 exemplos com problema:")
for registro in com_problema[:3]:
    print(
        f"  pedido_id={registro['pedido_id']}, "
        f"quantidade={registro['quantidade']!r}, "
        f"preco_unitario={registro['preco_unitario']!r}, "
        f"problemas={registro['_problemas']}"
    )

## Regras de negócio, datas e relatório final

Convertemos `data_venda`, testamos a hipótese sobre `data_entrega` vazia, contamos pedidos por canal e por **ano-mês**, e consolidamos o relatório.

In [ ]:
from functions import (
    analyze_entrega_sem_data,
    contar_pedidos_por_ano_mes,
    contar_pedidos_por_canal,
    gerar_relatorio_status,
    parse_data_venda,
)

entrega = analyze_entrega_sem_data(registros_limpos)
entrega.summary()

In [ ]:
print(
    "Hipótese: todo pedido sem data_entrega é Cancelado ->",
    "VERDADEIRA" if entrega.hipotese_todos_cancelados else "FALSA na base",
)
print(f"Sem entrega e cancelados: {entrega.sem_entrega_cancelados}")
print(f"Sem entrega com outro status: {entrega.sem_entrega_outros_status}\n")

for exemplo in entrega.exemplos_outros_status:
    print(
        f"  pedido_id={exemplo['pedido_id']}, status={exemplo['status_pedido']}, "
        f"data_entrega={exemplo['data_entrega']!r}"
    )

In [ ]:
exemplo_data = parse_data_venda(registros_limpos[0]["data_venda"])
exemplo_data

relatorio = gerar_relatorio_status(registros_limpos, stats_strings, stats_numericos)
relatorio.summary()

In [ ]:
canais = contar_pedidos_por_canal(registros_limpos)
periodos = contar_pedidos_por_ano_mes(registros_limpos)

canal_lider = max(canais.por_canal, key=canais.por_canal.get)
mes_lider = max(periodos.por_ano_mes, key=periodos.por_ano_mes.get)

print("Descoberta por canal:")
print(
    f"  {canal_lider} lidera com {canais.por_canal[canal_lider]} pedidos registrados "
    f"(excluídos por canal não reconhecido: {canais.excluidos})."
)
print("Descoberta por período:")
print(
    f"  {mes_lider} concentra {periodos.por_ano_mes[mes_lider]} pedidos com data_venda válida "
    f"(excluídos por data inválida: {periodos.excluidos})."
)